# Chapter 39 — Seams, Hooks, and Irreversible Decisions

**Companion to *Pi Agents*** · [`Pi Agents` chapter 39](https://programmer.ie/books/pi/39-chapter/)

| | |
|---|---|
| Chapter | `39-chapter.md` · weight 39 · `/books/pi/39-chapter/` |
| Notebook | `notebooks/pi/39-chapter.ipynb` |
| Pin | `@earendil-works/pi-*` **1.0.4** (`examples/package.json`) |
| Ground truth | `docs/*.md` and `dist/*.d.ts` **at the pin**, under `examples/node_modules/` |

> **Where the authority lives.** Every claim below is checked against the
> packages installed under `examples/node_modules/@earendil-works/`, not against
> a globally installed Pi and not from memory. When prose and an exported
> declaration disagree, the declaration wins.

## The question

In what order do the hooks fire — and does a block mean the body never ran,
while a rewrite cannot undo the effect?

The chapter's point: the hooks sit at named points in the run, in a **fixed
order**. `beforeToolCall` can **refuse before** the effect; `afterToolCall` can
**rewrite the result but cannot undo the effect**; `transformContext` changes
what the provider receives, not what the agent stored; `finishTurn` can end the
run early.


## What this notebook establishes

- The hooks fire in a **fixed order** in one tool-using run: prepareRequest → transformContext → convertToLlm → tool.execute starts → beforeToolCall → afterToolCall → finishTurn → turn_end.
- `beforeToolCall` can **refuse**, and then the effect never happens — `counter.runs` stays 0.
- `afterToolCall` can **rewrite the result but cannot undo the effect** — the write already happened.
- `transformContext` changes what the provider receives, **not what the agent stored**.
- `finishTurn` can **end the run early** (saving the follow-up request), the tool in the finished turn still ran.
- `prepareRequest` can **swap the model** for one request.
- `runToolCall` passes **nested calls through `beforeToolCall`**, and a blocked one never executes.

## What this notebook does **not** establish

- **Most seam-matrix cells are readings of declared result types, not runs** (`ch39-lim2`).
- **'Refuse' is relative to a named decision** — a hook can refuse the thing it is placed before, and nothing else.
- **The published 1.0.2 recording is inspected as historical**, not re-grounded.
- **The model is scripted** — request counts are exact for the scripted control flow only.


## Setup

This notebook runs the book's own examples. **Pi is Node/TypeScript**; Python
only presents, orchestrates and checks. There is no Python re-implementation
of Pi's loop, context loader, session projection, permissions or protocol
anywhere in this set — where a mechanism is demonstrated, the code that runs is
the code in `examples/`.

* **Execution mode:** **Run** + **Inspect** — the chapter's seam cases against a real `Agent`, a fresh trace from the chapter's exporter, and (when present) the published 1.0.2 recording.
* **Evidence scope:** `in_process_runtime`. The hooks are Pi's.
* **Credentials:** none. The model is a scripted stand-in (`faux`) supplied by
  the book's harness, so nothing here reaches the network, spends money, or
  says anything about how a real model behaves.


In [1]:
# --- Locate the shared support module and the pinned packages -----------------
# Two documented markers, both overridable:
#   PIN_REPO       -> the checkout of the pi repository
#   PIN_NOTEBOOKS  -> this notebook directory
# Because both searches walk upwards from the working directory, this cell
# works whether you launched Jupyter from notebooks/pi/ or from the repo root.
import sys
from pathlib import Path


def _find_support() -> Path:
    for c in (Path.cwd(), *Path.cwd().parents):
        if (c / "pinb.py").is_file():
            return c
        nested = c / "notebooks" / "pi"
        if (nested / "pinb.py").is_file():
            return nested
    raise SystemExit(
        "Could not find notebooks/pi/pinb.py from " + str(Path.cwd()) + ".\n"
        "Open this notebook from notebooks/pi/, or set PIN_NOTEBOOKS."
    )


sys.path.insert(0, str(_find_support()))
import pinb  # noqa: E402  (a local module, deliberately after the path fix)

pinb.self_check()


{'repo_root': 'resolved',
 'notebook_dir': 'resolved',
 'pin_matches_manuscript': True,
 'node': 'v25.8.2',
 'python': '3.13.15',
 'claim_ledger': True,
 'credentials_required': False}

In [2]:
pinb.show_environment("This notebook's runtime")

print()
print("Repository marker: an ancestor directory holding both")
print("  content/books/pi/                          (the manuscript)")
print("  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)")
print("Every path below is shown relative to that repository; no absolute")
print("path is printed, so the saved output is portable.")
print()
print(pinb.md_table(
    ["resource", "repository-relative path"],
    [["manuscript", pinb.rel(pinb.chapter_file(39))],
     ["examples tree", "examples/"],
     ["claim ledger", "examples/evidence.json"],
     ["pinned docs", "examples/node_modules/@earendil-works/pi-coding-agent/docs/"],
     ["pinned declarations", "examples/node_modules/@earendil-works/*/dist/"],
     ["support module", "notebooks/pi/pinb.py"]],
))


== This notebook's runtime ==
component  value
-------------------------------  ----------------------------------------------------
manuscript pin (AGENTS.md)  1.0.4
@earendil-works/pi-ai  1.0.4
@earendil-works/pi-agent-core  1.0.4
@earendil-works/pi-coding-agent  1.0.4
node  v25.8.2
python  3.13.15
platform  win32
examples tree  installed
model used  faux (scripted) - no network, no credential, no cost

Repository marker: an ancestor directory holding both
  content/books/pi/                          (the manuscript)
  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)
Every path below is shown relative to that repository; no absolute
path is printed, so the saved output is portable.

| resource | repository-relative path |
|---|---|
| manuscript | content/books/pi/39-chapter.md |
| examples tree | examples/ |
| claim ledger | examples/evidence.json |
| pinned docs | examples/node_modules/@earendil-works/pi-coding-agent/docs/ |
| pinned declarations | examples/node_mod

## Baseline: the hook order in a tool-using run

In [3]:
SEAMS = pinb.driver_source("ch39-seams.ts")
seams = pinb.driver(SEAMS, label="ch39-seams", timeout=180)
print("Hook order:")
for i, name in enumerate(seams["order"]["orderNames"], 1):
    print(f"  {i}. {name}")
print()
expected = ["prepareRequest", "transformContext", "convertToLlm", "tool.execute starts", "beforeToolCall", "afterToolCall", "finishTurn", "turn_end"]
first_turn = seams["order"]["orderNames"][:8]
pinb.show_checks([
    pinb.check("the first turn follows the documented hook order",
               first_turn == expected, " -> ".join(first_turn)),
    pinb.check("two turns cost two requests",
               seams["order"]["calls"] == 2, f"{seams['order']['calls']} requests"),
])

Hook order:
  1. prepareRequest
  2. transformContext
  3. convertToLlm
  4. tool.execute starts
  5. beforeToolCall
  6. afterToolCall
  7. finishTurn
  8. turn_end
  9. prepareRequest
  10. transformContext
  11. convertToLlm
  12. finishTurn
  13. turn_end

== Assertions ==
  assertion  observed
----  ------------------------------------------------  --------------------------------------------------------------------------
PASS  the first turn follows the documented hook order  prepareRequest -> transformContext -> convertToLlm -> tool.execute starts…
PASS  two turns cost two requests  2 requests

2/2 assertions held.


## Block before, rewrite after

In [4]:
bl, rw = seams["block"], seams["rewrite"]
print(pinb.md_table(
    ["case", "effect ran?", "tool result"],
    [["beforeToolCall blocks .env", f"{bl['runs']} runs", f"isError={bl['isError']}, {bl['text']}"],
     ["afterToolCall rewrites", f"{rw['runs']} runs", f"isError={rw['isError']}, {rw['text']}"]],
))

| case | effect ran? | tool result |
|---|---|---|
| beforeToolCall blocks .env | 0 runs | isError=True, protected path |
| afterToolCall rewrites | 1 runs | isError=True, [redacted] |


In [5]:
pinb.show_checks([
    pinb.check("a block means the effect never happened",
               bl["runs"] == 0 and bl["isError"] and "protected path" in bl["text"], "blocked, 0 runs"),
    pinb.check("a rewrite cannot undo the effect",
               rw["runs"] == 1, "1 run, already done"),
    pinb.check("the rewrite changed the result the model sees",
               rw["text"] == "[redacted]" and rw["isError"], "redacted"),
])

== Assertions ==
  assertion  observed
----  ---------------------------------------------  -------------------
PASS  a block means the effect never happened  blocked, 0 runs
PASS  a rewrite cannot undo the effect  1 run, already done
PASS  the rewrite changed the result the model sees  redacted

3/3 assertions held.


## transformContext and finishTurn

In [6]:
tc, fe = seams["transformContext"], seams["finishTurnEarly"]
print(pinb.md_table(
    ["check", "value"],
    [["sent user messages after transformContext", tc["sentUsers"]],
     ["stored user messages", tc["storedUsers"]],
     ["finishTurn end: requests", fe["callCount"]],
     ["finishTurn end: tool runs in the finished turn", fe["runs"]]],
))

| check | value |
|---|---|
| sent user messages after transformContext | 1 |
| stored user messages | 2 |
| finishTurn end: requests | 1 |
| finishTurn end: tool runs in the finished turn | 1 |


In [7]:
pinb.show_checks([
    pinb.check("transformContext changed what was sent, not what was stored",
               tc["sentUsers"] == 1 and tc["storedUsers"] == 2, "sent 1, stored 2"),
    pinb.check("finishTurn ended the run early (one request)",
               fe["callCount"] == 1, "1 request"),
    pinb.check("the tool in the finished turn still ran: finishTurn is after the effect",
               fe["runs"] == 1, "1 run"),
])

== Assertions ==
  assertion  observed
----  -----------------------------------------------------------------------  ----------------
PASS  transformContext changed what was sent, not what was stored  sent 1, stored 2
PASS  finishTurn ended the run early (one request)  1 request
PASS  the tool in the finished turn still ran: finishTurn is after the effect  1 run

3/3 assertions held.


## The published recording (historical, Pi 1.0.2)

The browser experience at `/tools/ai/pi/39-chapter/` replays a recording captured
on **Pi 1.0.2**. It is historical and stays labelled so.

In [8]:
import json
exp = pinb.experience_dir(39)
if exp is None:
    pinb.unrun("the published 1.0.2 recording", "not present in this checkout; set PIN_EXPERIENCES to the site repo's content/tools/ai/pi")
else:
    meta = json.loads((exp / "experiment.json").read_text(encoding="utf-8"))
    print("Published experience:", meta["id"])
    print("Pinned version:", meta["pinnedVersion"])
    print("RECORDED inspection of 1.0.2 evidence, not a fresh run.")

Published experience: pi/chapter-39/before-tool-call
Pinned version: 1.0.2
RECORDED inspection of 1.0.2 evidence, not a fresh run.


## The chapter's own tests

All eight seam tests.

## The chapter's canonical evidence

These are the chapter's own tests, executed unmodified through Node's test
runner. Every ledger row the chapter quotes comes from a named case in this
file, so running it here is running the chapter's evidence, not a copy of it.

In [9]:
PATTERNS = 'ch39-seams/seams.test.ts'
tap = pinb.run_tests(PATTERNS, timeout=300)
pinb.show_tap(tap, files='ch39-seams/seams.test.ts', show="beforeToolCall")

$ node --test ch39-seams/seams.test.ts
  PASS  8 passed, 0 failed, 0 skipped  in 366 ms

  tests matching 'beforeToolCall':
    ok   beforeToolCall can refuse, and then the effect never happens
    ok   runToolCall passes a nested call through beforeToolCall, and a blocked one never executes


## Your turn: predict, then run

**Predict first.** A block in `beforeToolCall` stops the effect. What does the
model see — a failed result, or nothing?

**Then change one input.** `afterToolCall` in the demonstration rewrites to
`isError: true`. Rewrite it to `isError: false` with the original text. Does the
effect count change?

**Predict the boundary.** The seam matrix distinguishes observe / transform /
refuse / effect-done. Which cell does `transformContext` occupy — and which does
`afterToolCall`?

In [10]:
print("Predictions:")
print("  1. A block in beforeToolCall: the model sees a failed toolResult with the reason.")
print("  2. Rewriting to isError false: the effect count does not change - it already ran.")
print("  3. transformContext transforms; afterToolCall is effect-done + transform.")
print()
print("Observed above:")
print(f"  order first turn: {" -> ".join(seams['order']['orderNames'][:8])}")
print(f"  block: runs={bl['runs']}, isError={bl['isError']}")
print(f"  rewrite: runs={rw['runs']}, result={rw['text']}")
print(f"  transformContext: sent={tc['sentUsers']}, stored={tc['storedUsers']}")
print()
assert bl["runs"] == 0 and rw["runs"] == 1
assert tc["sentUsers"] == 1 and tc["storedUsers"] == 2
assert fe["callCount"] == 1 and fe["runs"] == 1
print("held: hooks fire in order; a block precedes the effect; a rewrite cannot undo it")

Predictions:
  1. A block in beforeToolCall: the model sees a failed toolResult with the reason.
  2. Rewriting to isError false: the effect count does not change - it already ran.
  3. transformContext transforms; afterToolCall is effect-done + transform.

Observed above:
  order first turn: prepareRequest -> transformContext -> convertToLlm -> tool.execute starts -> beforeToolCall -> afterToolCall -> finishTurn -> turn_end
  block: runs=0, isError=True
  rewrite: runs=1, result=[redacted]
  transformContext: sent=1, stored=2

held: hooks fire in order; a block precedes the effect; a rewrite cannot undo it


## Interpretation

The seam matrix, with the cells this chapter's probes run:

| Hook | Observe | Transform | Refuse | Effect done? |
|---|---|---|---|---|
| `beforeToolCall` | yes | args | **block** | **no** — the effect never runs |
| `afterToolCall` | result | result | no | **yes** — cannot undo |
| `transformContext` | messages | messages | no | n/a (before the request) |
| `finishTurn` | turn | action | `end` | yes — after the effect |

Practical rules:
1. **Place the refusal before the effect.** `beforeToolCall` is the only gate that stops it.
2. **Do not let `afterToolCall` pretend to undo.** The write already happened; a rewrite changes only what the model sees.
3. **`transformContext` is a view, not storage.** What the agent stored differs from what the provider received.
4. **`finishTurn` after the effect** — the tool ran; only the follow-up request is saved.
5. **The recording is historical.** The 1.0.2 experience is inspected, not re-grounded.

## Sources, execution mode and limitations

**Execution mode:** **Run** + **Inspect** — the chapter's seam cases against a real `Agent`, a fresh trace from the chapter's exporter, and (when present) the published 1.0.2 recording.

**Evidence scope:** `in_process_runtime`. The hooks are Pi's.

### What was read or run

- **Ran** `drivers/ch39-seams.ts`: the chapter's order, block, rewrite, transformContext and finishTurn cases against a real `Agent`.
- **Ran** `ch39-seams/seams.test.ts` (8 tests).
- **Inspected** the published 1.0.2 recording at `/tools/ai/pi/39-chapter/` when `PIN_EXPERIENCES` resolves it (recorded, not fresh).
- **Read** `examples/evidence.json`, chapter 39 rows (8 claims).

### Limitations

- **Most seam-matrix cells are readings of declared result types, not runs** (`ch39-lim2`).
- **The published 1.0.2 recording is inspected as historical**, not re-grounded.

### Optional work that was NOT run

- The published browser experience is inspected only when its directory is present.

Every technical statement in this notebook is about Pi **1.0.4**. A later
release can change any of it; the book's drift protocol in `AGENTS.md` is the
procedure for finding out whether it did.
